<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Large_Moves_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Volatility Breakout Strategy & Daily Scanner

This notebook implements a technical analysis scanner designed to identify **Volatility Breakouts**. The strategy focuses on identifying stocks that experience significant price movements (up or down) relative to their historical volatility.

**Key Features:**
*   **Dynamic Signal Detection:** Identifies trigger days where the absolute daily return exceeds a multiple of the historical rolling volatility (configurable via `TARGET_MULTIPLIER`). Signals are detected for both 'up' and 'down' movements.
*   **Cooldown Mechanism:** Prevents redundant signals by enforcing a 'cool-off' period (`COOLDOWN_DAYS`) after a trigger is detected for a specific ticker in each direction.
*   **Ticker Loading:** Loads ticker symbols from a Google Drive file or falls back to a predefined list.
*   **Historical Data Fetching:** Downloads historical stock data using `yfinance`.
*   **Forward Performance Tracking:** Automatically calculates forward price changes (1-day through 5-day) *per share* following each breakout.
*   **Parameterized Retracement Analysis:** Analyzes the probability of price retracement, calculated as a percentage of the initial signal day's return, over the subsequent 5 days, customizable by a `TARGET_MULTIPLIER`.

In [38]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [39]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: df_symbols
Deleted DataFrame: raw_data
Deleted DataFrame: df_ticker
Deleted DataFrame: master_scanner_df
Deleted DataFrame: recent_signals_df
Deleted DataFrame: report_df
Deleted DataFrame: df_ticker_single
Deleted DataFrame: additional_tickers_report_all_data_df
All DataFrames cleared from memory.


In [40]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### Configure Scanner Parameters

In [41]:
# ---- SCANNER PARAMETERS ----
LOOKBACK_CALENDAR_DAYS = 100  # Pull enough data for the 60-day volatility window
SCAN_SIGNAL_DAYS = 20         # Filter final report to signals from the last 20 trading days
VOL_WINDOW = 60               # Rolling window for volatility
COOLDOWN_DAYS = 5             # Days to ignore new signals after a trigger
TARGET_MULTIPLIER = 3       # The breakout extreme we want to track profitability for

# Dynamically calculate window dates based on today
end_dt = datetime.today() + timedelta(days=1)
start_dt = end_dt - timedelta(days=LOOKBACK_CALENDAR_DAYS)

START_DATE = start_dt.strftime("%Y-%m-%d")
END_DATE = end_dt.strftime("%Y-%m-%d")

print(f"Scanner configured to pull historical data from {START_DATE} through today ({END_DATE}).")

Scanner configured to pull historical data from 2026-04-09 through today (2026-07-18).


### Load Tickers & Fetch Live Data

In [42]:
fallback_tickers = ["AAPL", "MSFT", "NVDA", "AMD", "TSLA"]

additional_tickers = ["ISRG"] # Add your desired tickers here
print(f"Custom tickers to be added: {additional_tickers}")

try:
    print("Attempting to pull tickers from the Option Volume file...")
    df_symbols = pd.read_csv(OptionVolume200)
    if 'Symbol' in df_symbols.columns:
        tickers = df_symbols['Symbol'].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        print(f"Successfully loaded {len(tickers)} tickers from OptionVolume200.")
    else:
        raise KeyError("'Symbol' column was not found in the downloaded file.")
except Exception as e:
    print(f"Failed to load tickers from Google Drive ({e}). Falling back to default list.")
    tickers = fallback_tickers

print(f"Fetching data block for {len(tickers)} tickers...")
# yf.download handles parsing the live intraday bar as today's 'Close' automatically
raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, group_by='ticker')

Custom tickers to be added: ['ISRG']
Attempting to pull tickers from the Option Volume file...
Successfully loaded 200 tickers from OptionVolume200.
Fetching data block for 200 tickers...


/tmp/ipykernel_1177/1572724012.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, group_by='ticker')
[*********************100%***********************]  200 of 200 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SATS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-09 -> 2026-07-18)')


In [43]:
fallback_tickers = ["AAPL", "MSFT", "NVDA", "AMD", "TSLA"]

try:
    print("Attempting to pull tickers from the Option Volume file...")
    df_symbols = pd.read_csv(OptionVolume200)
    if 'Symbol' in df_symbols.columns:
        tickers = df_symbols['Symbol'].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        print(f"Successfully loaded {len(tickers)} tickers from OptionVolume200.")
    else:
        raise KeyError("'Symbol' column was not found in the downloaded file.")
except Exception as e:
    print(f"Failed to load tickers from Google Drive ({e}). Falling back to default list.")
    tickers = fallback_tickers

# Combine with user-provided additional tickers and remove duplicates
combined_tickers = list(set(tickers + additional_tickers))

print(f"Fetching data block for {len(combined_tickers)} tickers...")
# yf.download handles parsing the live intraday bar as today's 'Close' automatically
raw_data = yf.download(combined_tickers, start=START_DATE, end=END_DATE, group_by='ticker')

Attempting to pull tickers from the Option Volume file...


/tmp/ipykernel_1177/1923539737.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(combined_tickers, start=START_DATE, end=END_DATE, group_by='ticker')
[                       0%                       ]

Successfully loaded 200 tickers from OptionVolume200.
Fetching data block for 201 tickers...


[*********************100%***********************]  201 of 201 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SATS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-04-09 -> 2026-07-18)')


### Live Signal Scan & Dollar Profitability Processing

In [44]:
scanner_results = []

for ticker in combined_tickers:
    if ticker not in raw_data.columns.levels[0]:
        continue

    df_ticker = raw_data[ticker].dropna(subset=['Close']).copy()
    if len(df_ticker) < VOL_WINDOW:
        continue

    # Standard metrics
    df_ticker['Daily_Return'] = df_ticker['Close'].pct_change()
    df_ticker['Return_Mag'] = df_ticker['Daily_Return'].abs()
    df_ticker['Prev_Close'] = df_ticker['Close'].shift(1)
    df_ticker['Hist_Vol'] = df_ticker['Daily_Return'].rolling(window=VOL_WINDOW).std()
    df_ticker['Vol_Threshold'] = df_ticker['Hist_Vol'] * TARGET_MULTIPLIER

    # Process signals tracking cooldowns
    cooldown_counter_up = 0
    cooldown_counter_down = 0
    trigger_rows = []

    # Iterate through the DataFrame to find signals for 'up' and 'down' directions
    # Each direction gets its own cooldown counter
    for idx, row in df_ticker.iterrows():
        # Check for 'up' signals
        if cooldown_counter_up == 0 and row['Daily_Return'] > 0 and row['Return_Mag'] > row['Vol_Threshold']:
            trigger_rows.append((idx, 'up'))
            cooldown_counter_up = COOLDOWN_DAYS
        elif cooldown_counter_up > 0:
            cooldown_counter_up -= 1

        # Check for 'down' signals
        if cooldown_counter_down == 0 and row['Daily_Return'] < 0 and row['Return_Mag'] > row['Vol_Threshold']:
            trigger_rows.append((idx, 'down'))
            cooldown_counter_down = COOLDOWN_DAYS
        elif cooldown_counter_down > 0:
            cooldown_counter_down -= 1

    # Calculate forward dollar values for verified signals
    for idx, direction in trigger_rows:
        try:
            loc = df_ticker.index.get_loc(idx)
            day_0_close = df_ticker.iloc[loc]['Close']
            day_0_ret = df_ticker.iloc[loc]['Daily_Return']
            day_0_return_mag = df_ticker.iloc[loc]['Return_Mag'] # Get Return_Mag for the signal day
            day_0_hist_vol = df_ticker.iloc[loc]['Hist_Vol'] # Get Hist_Vol for the signal day

            # Since MAX_ALLOCATION is removed, dollar_profit will represent profit/loss per share
            # No 'shares' calculation needed, directly use price_diff

            row_data = {
                'Date': idx.strftime("%Y-%m-%d"),
                'Symbol': ticker,
                'Price': round(day_0_close, 2),
                'Direction': direction,
                'Multiplier': TARGET_MULTIPLIER, # Re-adding Multiplier
                'Daily_Return': day_0_ret,
                'Return_Mag': day_0_return_mag,
                'Hist_Vol': day_0_hist_vol # Add Hist_Vol to row_data
            }

            for d in range(1, 6):
                if loc + d < len(df_ticker):
                    fwd_close = df_ticker.iloc[loc + d]['Close']
                    price_diff = fwd_close - day_0_close
                    # Store price_diff directly as 'dollar_profit' (per share)
                    dollar_profit = price_diff if direction == 'up' else -price_diff
                    row_data[f'Day_{d}_$'] = round(dollar_profit, 2)
                else:
                    row_data[f'Day_{d}_$'] = np.nan

            scanner_results.append(row_data)
        except Exception:
            continue

# Compile Master Scanner Output
if scanner_results:
    master_scanner_df = pd.DataFrame(scanner_results)
    master_scanner_df['Datetime'] = pd.to_datetime(master_scanner_df['Date'])
    master_scanner_df = master_scanner_df.sort_values(by='Datetime', ascending=False)

    # NEW LOGIC: Filter based on an actual rolling window of the last 20 trading days
    # We find the unique trading dates in our downloaded dataset and look at the last 20
    available_dates = sorted(master_scanner_df['Datetime'].unique())
    cutoff_idx = max(0, len(available_dates) - SCAN_SIGNAL_DAYS)
    cutoff_date = available_dates[cutoff_idx]

    recent_signals_df = master_scanner_df[master_scanner_df['Datetime'] >= cutoff_date].copy()
    recent_signals_df = recent_signals_df.drop(columns=['Datetime'])

    total_found = len(recent_signals_df)
    if total_found > 0:
        print(f"Scan Complete: Processed {total_found} signals found within the true last {SCAN_SIGNAL_DAYS} trading days ({cutoff_date.strftime('%Y-%m-%d')} to today).")
    else:
        print(f"Scan Complete: 0 signals matched parameters within the last {SCAN_SIGNAL_DAYS} trading days.")
else:
    recent_signals_df = pd.DataFrame()
    print("Scan Complete: No signals generated across the historical timeline.")

Scan Complete: Processed 10 signals found within the true last 20 trading days (2026-07-08 to today).


### Generate Scanned Profitability Report

In [45]:
# ==========================================
# 5. GENERATE SCANNED RETRACEMENT REPORT
# ==========================================

print("📊" * 25)
print(f"CLOSING RETRACEMENT REPORT ({TARGET_MULTIPLIER}x Multiplier) — LAST {SCAN_SIGNAL_DAYS} SIGNALS")
print("📊" * 25)

if not recent_signals_df.empty:
    # Create a clean reporting copy to calculate percentages without mutating master data
    report_df = recent_signals_df.copy()

    # The 'Return_Mag' column is now directly used for display.
    # The previous line `report_df['Multiplier'] = TARGET_MULTIPLIER` is removed as it's no longer needed.

    # Add Vol_Ratio column
    report_df['Vol_Ratio'] = report_df['Return_Mag'] / report_df['Hist_Vol']

    for d in range(1, 6):
        dollar_col = f'Day_{d}_$'
        pct_col = f'Retrace_Day_{d}'

        # Absolute magnitude of the Day 0 baseline move
        impulse_size_pct = report_df['Daily_Return'].abs()

        # Percentage change from Day 0 close
        # Since dollar_col now contains the raw dollar difference (price_diff) per share,
        # to get the forward return, we divide by the initial price (report_df['Price']).
        fwd_return_from_close = report_df[dollar_col] / report_df['Price']

        # Retracement % = -(Forward Return from Day 0 Close) / (Day 0 Return)
        report_df[pct_col] = -fwd_return_from_close / impulse_size_pct

    # Change 'Multiplier' to 'Return_Mag' in the display columns list
    display_cols = ['Date', 'Symbol', 'Price', 'Direction', 'Return_Mag', 'Hist_Vol', 'Vol_Ratio', 'Retrace_Day_1', 'Retrace_Day_2', 'Retrace_Day_3', 'Retrace_Day_4', 'Retrace_Day_5']

    # Format floating numbers as percentages, leaving NaNs completely intact
    print(report_df[display_cols].to_string(index=False, na_rep='NaN', formatters={
        'Price': '${:,.2f}'.format,
        'Return_Mag': '{:,.1%}'.format, # Format 'Return_Mag' as percentage
        'Hist_Vol': '{:,.1%}'.format, # Format 'Hist_Vol' as percentage
        'Vol_Ratio': '{:,.1f}x'.format, # Format 'Vol_Ratio' as a multiplier
        'Retrace_Day_1': '{:,.1%}'.format, 'Retrace_Day_2': '{:,.1%}'.format,
        'Retrace_Day_3': '{:,.1%}'.format, 'Retrace_Day_4': '{:,.1%}'.format, 'Retrace_Day_5': '{:,.1%}'.format
    }))

    print("\n--- Retracement >= 100% Counts and Ratios ---")
    retracement_counts = {}
    retracement_ratios = {}

    for d in range(1, 6):
        col_name = f'Retrace_Day_{d}'
        # Count signals where retracement is 100% or more (value >= 1.0)
        count_100_percent = (report_df[col_name] >= 1.0).sum()
        # Count total valid (non-NaN) signals for the day
        total_valid_signals = report_df[col_name].count()

        retracement_counts[col_name] = count_100_percent
        if total_valid_signals > 0:
            retracement_ratios[col_name] = count_100_percent / total_valid_signals
        else:
            retracement_ratios[col_name] = np.nan # No valid signals to calculate ratio

    # Prepare for printing in a table-like format
    header = "" \
             + "{:<15}".format("Metric")
    for d in range(1, 6):
        header += "{:<15}".format(f'Day {d}')
    print(header)

    counts_row = "" \
                 + "{:<15}".format("Count (>=100%)")
    for d in range(1, 6):
        counts_row += "{:<15}".format(str(retracement_counts[f'Retrace_Day_{d}'] if f'Retrace_Day_{d}' in retracement_counts else 0))
    print(counts_row)

    ratios_row = "" \
                 + "{:<15}".format("Ratio (>=100%)")
    for d in range(1, 6):
        ratio_val = retracement_ratios.get(f'Retrace_Day_{d}')
        if pd.isna(ratio_val):
            ratios_row += "{:<15}".format("N/A")
        else:
            ratios_row += "{:<15}".format(f'{ratio_val:.1%}')
    print(ratios_row)

else:
    print(f"No breakout events crossed the {TARGET_MULTIPLIER}x standard deviation threshold in the last 20 days.")

📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
CLOSING RETRACEMENT REPORT (3x Multiplier) — LAST 20 SIGNALS
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
      Date Symbol     Price Direction Return_Mag Hist_Vol Vol_Ratio Retrace_Day_1 Retrace_Day_2 Retrace_Day_3 Retrace_Day_4 Retrace_Day_5
2026-07-17   NFLX    $68.70      down       7.6%     2.1%      3.7x           NaN           NaN           NaN           NaN           NaN
2026-07-17   ISRG   $349.24      down      13.2%     3.0%      4.5x           NaN           NaN           NaN           NaN           NaN
2026-07-16    ABT    $98.83        up      10.7%     2.3%      4.8x        -10.6%           NaN           NaN           NaN           NaN
2026-07-15   PYPL    $55.52        up      17.2%     3.0%      5.7x        -12.7%        -14.7%           NaN           NaN           NaN
2026-07-15    PGR   $205.22      down       9.4%     2.1%      4.5x          3.0%         12.0%           NaN           NaN           NaN
2026-07-14   CRWD   $210.73        up      12.1%     3.5%  

### Latest Metrics for `additional_tickers` (All Data)

In [46]:
import pandas as pd
import numpy as np

results_for_additional_tickers = []

# Ensure `additional_tickers` is defined and not empty
if 'additional_tickers' in globals() and additional_tickers:
    for ticker in additional_tickers:
        # Check if the ticker exists in the raw_data
        if ticker in raw_data.columns.levels[0]:
            df_ticker_single = raw_data[ticker].dropna(subset=['Close']).copy()

            # Ensure enough data for volatility calculation
            if len(df_ticker_single) >= VOL_WINDOW:
                # Calculate Daily Return
                df_ticker_single['Daily_Return'] = df_ticker_single['Close'].pct_change()

                # Calculate Historical Volatility (rolling std)
                df_ticker_single['Hist_Vol'] = df_ticker_single['Daily_Return'].rolling(window=VOL_WINDOW).std()

                # Get the latest available day's data
                latest_day_data = df_ticker_single.iloc[-1]

                # Extract metrics for the latest day
                latest_price = latest_day_data['Close']
                latest_daily_return = latest_day_data['Daily_Return']
                latest_return_mag = abs(latest_daily_return) if not pd.isna(latest_daily_return) else np.nan
                latest_hist_vol = latest_day_data['Hist_Vol']

                # Calculate Volatility Ratio
                latest_vol_ratio = latest_return_mag / latest_hist_vol if latest_hist_vol != 0 and not pd.isna(latest_return_mag) else np.nan

                # Determine Direction
                if not pd.isna(latest_daily_return):
                    direction = 'up' if latest_daily_return > 0 else 'down' if latest_daily_return < 0 else 'flat'
                else:
                    direction = 'N/A'

                # Check for signal based on current TARGET_MULTIPLIER
                signal_triggered = False
                if not pd.isna(latest_return_mag) and not pd.isna(latest_hist_vol) and latest_hist_vol > 0:
                    if latest_return_mag > (latest_hist_vol * TARGET_MULTIPLIER):
                        signal_triggered = True

                results_for_additional_tickers.append({
                    'Symbol': ticker,
                    'Date': df_ticker_single.index[-1].strftime("%Y-%m-%d") if not df_ticker_single.empty else 'N/A',
                    'Price': latest_price,
                    'Direction': direction,
                    'Return_Mag': latest_return_mag,
                    'Hist_Vol': latest_hist_vol,
                    'Vol_Ratio': latest_vol_ratio,
                    'Signal_Triggered': 'Yes' if signal_triggered else 'No'
                })
            else:
                results_for_additional_tickers.append({
                    'Symbol': ticker,
                    'Date': 'N/A',
                    'Price': np.nan,
                    'Direction': 'N/A',
                    'Return_Mag': np.nan,
                    'Hist_Vol': np.nan,
                    'Vol_Ratio': np.nan,
                    'Signal_Triggered': 'Not enough data for volatility calculation'
                })
        else:
            results_for_additional_tickers.append({
                'Symbol': ticker,
                'Date': 'N/A',
                'Price': np.nan,
                'Direction': 'N/A',
                'Return_Mag': np.nan,
                'Hist_Vol': np.nan,
                'Vol_Ratio': np.nan,
                'Signal_Triggered': 'No data found in raw_data'
            })

    if results_for_additional_tickers:
        additional_tickers_report_all_data_df = pd.DataFrame(results_for_additional_tickers)
        print("\n--- Latest Metrics for additional_tickers (regardless of signal) ---")
        display_cols = ['Symbol', 'Date', 'Price', 'Direction', 'Return_Mag', 'Hist_Vol', 'Vol_Ratio', 'Signal_Triggered']

        print(additional_tickers_report_all_data_df[display_cols].to_string(index=False, na_rep='NaN', formatters={
            'Price': '${:,.2f}'.format,
            'Return_Mag': '{:,.1%}'.format,
            'Hist_Vol': '{:,.1%}'.format,
            'Vol_Ratio': '{:,.1f}x'.format
        }))
    else:
        print("No results to display for `additional_tickers`.")
else:
    print("No additional tickers specified or the `additional_tickers` list is empty.")


--- Latest Metrics for additional_tickers (regardless of signal) ---
Symbol       Date   Price Direction Return_Mag Hist_Vol Vol_Ratio Signal_Triggered
  ISRG 2026-07-17 $349.24      down      13.2%     3.0%      4.5x              Yes
